In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf

# Define the ticker symbol and download historical data for 5 years
ticker = "0005.HK"
data = yf.Ticker(ticker).history(period="5y")

# Reset the index so that Date becomes a column
data.reset_index(inplace=True)

# Convert Date column to display only the date (without time)


# --- Technical Indicator Calculations ---

# SMA_20: 20-day Simple Moving Average
data['SMA_20'] = data['Close'].rolling(window=20).mean()

# EMA_20: 20-day Exponential Moving Average
data['EMA_20'] = data['Close'].ewm(span=20, adjust=False).mean()

# Moving_Average: Average of SMA_20 and EMA_20
data['Moving_Average'] = (data['SMA_20'] + data['EMA_20']) / 2

# ROC: Rate of Change over a 12-day period (percentage)
data['ROC'] = data['Close'].pct_change(periods=12) * 100

# MACD: Difference between 12-day EMA and 26-day EMA
EMA_12 = data['Close'].ewm(span=12, adjust=False).mean()
EMA_26 = data['Close'].ewm(span=26, adjust=False).mean()
data['MACD'] = EMA_12 - EMA_26

# Daily Price Returns: Daily percentage change in closing price
data['Daily_Returns'] = data['Close'].pct_change() * 100

# VFI (Volume Flow Indicator): Using a simplified approach as a volume z-score
data['VFI'] = (data['Volume'] - data['Volume'].rolling(window=20).mean()) / data['Volume'].rolling(window=20).std()

# MFI (Money Flow Index) Calculation:
data['Typical_Price'] = (data['High'] + data['Low'] + data['Close']) / 3
data['Raw_Money_Flow'] = data['Typical_Price'] * data['Volume']
data['Positive_MF'] = np.where(data['Typical_Price'].diff() > 0, data['Raw_Money_Flow'], 0)
data['Negative_MF'] = np.where(data['Typical_Price'].diff() < 0, data['Raw_Money_Flow'], 0)
data['Money_Ratio'] = data['Positive_MF'].rolling(14).sum() / data['Negative_MF'].rolling(14).sum().replace(0, np.nan)
data['MFI'] = 100 - (100 / (1 + data['Money_Ratio']))

# --- Prepare final DataFrame ---
final_columns = [
    'Date', 'Close', 'Volume', 'SMA_20', 'EMA_20', 'Moving_Average',
    'ROC', 'MACD', 'Daily_Returns', 'VFI', 'MFI'
]
data_final = data[final_columns]

# Print first five rows for verification
print(data_final.head())

# --- Save DataFrame to Excel ---
excel_path = r"D:\Steven Documents\Project\HK options stock analysis\Stock data\0005.xlsx"
data_final.to_excel(excel_path, index=False, sheet_name="Stock Data")

print("Data fetching and processing complete. Excel file saved at:", excel_path)


                       Date      Close    Volume  SMA_20     EMA_20  \
0 2020-05-26 00:00:00+08:00  29.318176  33450161     NaN  29.318176   
1 2020-05-27 00:00:00+08:00  29.675718  42822281     NaN  29.352228   
2 2020-05-28 00:00:00+08:00  29.397631  56105691     NaN  29.356552   
3 2020-05-29 00:00:00+08:00  28.483921  41062941     NaN  29.273444   
4 2020-06-01 00:00:00+08:00  29.079817  50234471     NaN  29.255004   

   Moving_Average  ROC      MACD  Daily_Returns  VFI  MFI  
0             NaN  NaN  0.000000            NaN  NaN  NaN  
1             NaN  NaN  0.028522       1.219523  NaN  NaN  
2             NaN  NaN  0.028359      -0.937088  NaN  NaN  
3             NaN  NaN -0.044980      -3.108106  NaN  NaN  
4             NaN  NaN -0.054390       2.092043  NaN  NaN  


OSError: Cannot save file into a non-existent directory: 'D:\Steven Documents\Project\HK options stock analysis\Stock data'

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf

# List of stock tickers to fetch
stock_tickers = [
    "0939.HK","0388.HK"
]

# Define save location
save_directory = r"D:\Steven Documents\Project\HK options stock analysis\Stock data"

# Function to fetch and process stock data
def fetch_and_save_stock_data(ticker):
    print(f"Fetching data for {ticker}...")
    
    # Fetch historical data (5 years)
    data = yf.Ticker(ticker).history(period="5y")
    
    # Reset index and format date
    data.reset_index(inplace=True)
    data["Date"] = data["Date"].dt.date

    # Compute technical indicators
    data['SMA_20'] = data['Close'].rolling(window=20).mean()
    data['EMA_20'] = data['Close'].ewm(span=20, adjust=False).mean()
    data['Moving_Average'] = (data['SMA_20'] + data['EMA_20']) / 2
    data['ROC'] = data['Close'].pct_change(periods=12) * 100

    # MACD calculation
    EMA_12 = data['Close'].ewm(span=12, adjust=False).mean()
    EMA_26 = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD'] = EMA_12 - EMA_26

    data['Daily_Returns'] = data['Close'].pct_change() * 100

    # Volume Flow Indicator (VFI)
    data['VFI'] = (data['Volume'] - data['Volume'].rolling(window=20).mean()) / data['Volume'].rolling(window=20).std()

    # Money Flow Index (MFI)
    data['Typical_Price'] = (data['High'] + data['Low'] + data['Close']) / 3
    data['Raw_Money_Flow'] = data['Typical_Price'] * data['Volume']
    data['Positive_MF'] = np.where(data['Typical_Price'].diff() > 0, data['Raw_Money_Flow'], 0)
    data['Negative_MF'] = np.where(data['Typical_Price'].diff() < 0, data['Raw_Money_Flow'], 0)
    data['Money_Ratio'] = data['Positive_MF'].rolling(14).sum() / data['Negative_MF'].rolling(14).sum().replace(0, np.nan)
    data['MFI'] = 100 - (100 / (1 + data['Money_Ratio']))

    # Select final columns
    final_columns = ['Date', 'Close', 'Volume', 'SMA_20', 'EMA_20', 'Moving_Average', 'ROC', 'MACD', 'Daily_Returns', 'VFI', 'MFI']
    data_final = data[final_columns]

    # Define file path and save to Excel
    file_name = f"{ticker.split('.')[0]}.xlsx"  # Extract only stock number from ticker (e.g., 9988.HK → 9988.xlsx)
    file_path = f"{save_directory}\\{file_name}"
    data_final.to_excel(file_path, index=False, sheet_name="Stock Data")
    
    print(f"Saved {file_name} successfully.")

# Loop through stock tickers and process each one
for stock in stock_tickers:
    fetch_and_save_stock_data(stock)

print("All stock data fetching and saving complete!")


Fetching data for 0939.HK...
Saved 0939.xlsx successfully.
Fetching data for 0388.HK...
Saved 0388.xlsx successfully.
All stock data fetching and saving complete!
